In [1]:
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Attention from scratch
Goal: understand self-attention by solving the "first digit" memory task
that RNN / LSTM / CNN struggled with.

In [2]:
def make_memory_data(n, length=30):
    """Random digits 0-9; label is 1 if the FIRST digit is > 4."""
    x = torch.randint(0, 10, (n, length))   # (n, length)
    y = (x[:, 0] > 4).long()                # (n,)
    return x, y

In [5]:
x, y = make_memory_data(5)
print(x.shape, y.shape)
print(x[:, 0])
print(y)

torch.Size([5, 30]) torch.Size([5])
tensor([6, 6, 7, 2, 1])
tensor([1, 1, 1, 0, 0])


## A2. Attention by hand
Each word compares itself with every word (dot product),
softmax turns scores into weights (sum = 1),
the new vector is a weighted average of all vectors.

In [7]:
def toy_attention(x, idx):
    """Attention for ONE word (row idx of x) over all words in x."""
    scores = x @ x[idx]                      # similarity with every word, shape (n_words,)
    weights = torch.softmax(scores, dim=0)   # weights sum to 1
    new_vec = weights @ x                    # weighted average of all vectors
    return scores, weights, new_vec

# toy vectors: [how "alive", how "place/object"]
words = ["kot", "ulice", "byl"]
x_toy = torch.tensor([[1.0, 0.1],    # kot
                      [0.1, 1.0],    # ulice
                      [0.9, 0.2]])   # byl: looks for something alive

scores, weights, new_vec = toy_attention(x_toy, idx=2)
print("scores :", scores)
print("weights:", weights, "sum =", weights.sum().item())
print("new byl:", new_vec)

scores : tensor([0.9200, 0.2900, 0.8500])
weights: tensor([0.4057, 0.2161, 0.3783]) sum = 1.0
new byl: tensor([0.7677, 0.3323])


## A3. Self-attention with learned Q, K, V
Each word is projected three times (Linear layers):
- Q (query): what am I looking for?
- K (key): what do I offer as a label?
- V (value): what do I pass on if someone attends to me?
scores = Q K^T / sqrt(d) -> softmax -> weighted average of V.

In [8]:
def self_attention(x, w_q, w_k, w_v):
    """Single-head self-attention. x has shape (n_words, d)."""
    q = w_q(x)                          # (n_words, d) what each word is looking for
    k = w_k(x)                          # (n_words, d) label each word offers
    v = w_v(x)                          # (n_words, d) content each word passes on
    d = q.shape[-1]
    scores = q @ k.T / math.sqrt(d)     # (n_words, n_words) everyone vs everyone
    weights = torch.softmax(scores, dim=-1)   # each ROW sums to 1
    return weights @ v, weights         # new vectors (n_words, d) and the weights


torch.manual_seed(0)
x = torch.randn(3, 4)                   # 3 words, 4 numbers each (random for now)
w_q, w_k, w_v = (nn.Linear(4, 4, bias=False) for _ in range(3))

out, weights = self_attention(x, w_q, w_k, w_v)
print("out shape    :", out.shape)
print("weights shape:", weights.shape)
print(weights)
print("row sums     :", weights.sum(dim=-1))

out shape    : torch.Size([3, 4])
weights shape: torch.Size([3, 3])
tensor([[0.3484, 0.3430, 0.3087],
        [0.1911, 0.4861, 0.3228],
        [0.2104, 0.4680, 0.3216]], grad_fn=<SoftmaxBackward0>)
row sums     : tensor([1., 1., 1.], grad_fn=<SumBackward1>)


## A4. Attention on the "first digit" task
Embedding + position -> self-attention -> read the LAST token -> Linear.
If it works, the last token must learn to look at position 0.

In [9]:
class AttentionNet(nn.Module):
    """Embedding (+ position) -> one self-attention layer -> classify from LAST token."""

    def __init__(self, vocab_size=10, d=32, length=10, use_pos=True):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d)
        self.pos = nn.Embedding(length, d) if use_pos else None   # which position is it?
        self.w_q = nn.Linear(d, d, bias=False)
        self.w_k = nn.Linear(d, d, bias=False)
        self.w_v = nn.Linear(d, d, bias=False)
        self.fc = nn.Linear(d, 2)
        self.last_weights = None                                   # saved for inspection

    def forward(self, x):                                          # x: (batch, length)
        h = self.emb(x)                                            # (batch, length, d)
        if self.pos is not None:
            h = h + self.pos(torch.arange(x.shape[1], device=x.device))
        q, k, v = self.w_q(h), self.w_k(h), self.w_v(h)            # (batch, length, d)
        scores = q @ k.transpose(1, 2) / math.sqrt(q.shape[-1])    # (batch, length, length)
        weights = torch.softmax(scores, dim=-1)
        self.last_weights = weights.detach()
        out = weights @ v                                          # (batch, length, d)
        return self.fc(out[:, -1])                                 # only the last token


def make_loaders(n_train=20000, n_val=2000, length=10, batch_size=64):
    x_tr, y_tr = make_memory_data(n_train, length)
    x_va, y_va = make_memory_data(n_val, length)
    train_loader = DataLoader(TensorDataset(x_tr, y_tr), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(x_va, y_va), batch_size=batch_size)
    return train_loader, val_loader


def run_epoch(model, loader, optimizer=None):
    """One pass over the data; trains if an optimizer is given. Returns (loss, acc)."""
    training = optimizer is not None
    model.train(training)
    loss_fn = nn.CrossEntropyLoss()
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(yb)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            n += len(yb)
    return total_loss / n, correct / n


def train_attention(use_pos=True, length=10, epochs=10, lr=3e-3):
    torch.manual_seed(42)
    train_loader, val_loader = make_loaders(length=length)
    model = AttentionNet(length=length, use_pos=use_pos).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = run_epoch(model, train_loader, optimizer)
        va_loss, va_acc = run_epoch(model, val_loader)
        print(f"Epoch {epoch}/{epochs} | train acc {tr_acc:.3f} | val loss {va_loss:.3f} acc {va_acc:.3f}")
    return model, val_loader


model, val_loader = train_attention(use_pos=True)

# Where does the LAST token look? (weights of the last row, first validation sample)
xb, yb = next(iter(val_loader))
model.eval()
model(xb.to(device))
torch.set_printoptions(precision=2, sci_mode=False)
print(xb[0])
print(model.last_weights[0, -1])

Epoch 1/10 | train acc 0.971 | val loss 0.000 acc 1.000
Epoch 2/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 3/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 4/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 5/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 6/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 7/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 8/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 9/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 10/10 | train acc 1.000 | val loss 0.000 acc 1.000
tensor([7, 1, 3, 0, 6, 2, 9, 1, 2, 1])
tensor([1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00])


## A5. Positional embedding: why attention needs it

Self-attention treats the input as a *set*: it has no idea which digit came first.
Fix: add a learned vector per position, `h = emb(digit) + pos(position)`.

**Experiment:** same model, same data (length 10), with and without `pos`.

| model | val acc |
|---|---|
| attention + pos | 1.000 |
| attention, no pos | ~0.62 |

Why ~0.62 and not 0.5? Without order the model still sees *how many* digits are > 4.
The first digit is one of them, so more big digits make "first digit is big" more likely.

In [11]:
model_no_pos, _ = train_attention(use_pos=False)

Epoch 1/10 | train acc 0.615 | val loss 0.650 acc 0.608
Epoch 2/10 | train acc 0.628 | val loss 0.640 acc 0.614
Epoch 3/10 | train acc 0.630 | val loss 0.638 acc 0.623
Epoch 4/10 | train acc 0.631 | val loss 0.638 acc 0.624
Epoch 5/10 | train acc 0.629 | val loss 0.639 acc 0.619
Epoch 6/10 | train acc 0.632 | val loss 0.637 acc 0.619
Epoch 7/10 | train acc 0.631 | val loss 0.638 acc 0.622
Epoch 8/10 | train acc 0.629 | val loss 0.639 acc 0.618
Epoch 9/10 | train acc 0.631 | val loss 0.634 acc 0.622
Epoch 10/10 | train acc 0.632 | val loss 0.638 acc 0.626


In [13]:
class BaselineNet(nn.Module):
    """Embedding -> RNN / LSTM / CNN -> Linear(2). Same task and data as AttentionNet."""

    def __init__(self, kind, vocab_size=10, d=32):
        super().__init__()
        self.kind = kind
        self.emb = nn.Embedding(vocab_size, d)
        if kind == "rnn":
            self.core = nn.RNN(d, d, batch_first=True)
        elif kind == "lstm":
            self.core = nn.LSTM(d, d, batch_first=True)
        elif kind == "cnn":
            self.core = nn.Conv1d(d, d, kernel_size=3, padding=1)
        else:
            raise ValueError(kind)
        self.fc = nn.Linear(d, 2)

    def forward(self, x):                                  # x: (batch, length)
        h = self.emb(x)                                    # (batch, length, d)
        if self.kind == "cnn":
            h = torch.relu(self.core(h.transpose(1, 2)))   # Conv1d wants (batch, d, length)
            return self.fc(h.max(dim=2).values)            # global max pooling over positions
        out, _ = self.core(h)                              # (batch, length, d)
        return self.fc(out[:, -1])                         # last time step only


def set_forget_bias(lstm, value=2.0):
    """Set the forget-gate bias. PyTorch bias layout is [input | forget | cell | output]."""
    d = lstm.hidden_size
    with torch.no_grad():
        lstm.bias_ih_l0[d:2 * d].fill_(value)


def train_model(model, length, epochs=10, lr=3e-3, n_train=50000):
    """Train any of our models and return the best validation accuracy."""
    train_loader, val_loader = make_loaders(n_train=n_train, length=length)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_acc = 0.0
    for _ in range(epochs):
        run_epoch(model, train_loader, optimizer)
        _, val_acc = run_epoch(model, val_loader)
        best_acc = max(best_acc, val_acc)
    return best_acc


def compare_models(length=30, epochs=10):
    """Train every model on the same task and print the best val accuracy of each."""
    results = {}
    for name in ("rnn", "lstm", "lstm_bias", "cnn", "attention"):
        torch.manual_seed(42)
        if name == "attention":
            model = AttentionNet(length=length)
        else:
            model = BaselineNet("lstm" if name == "lstm_bias" else name)
        if name == "lstm_bias":
            set_forget_bias(model.core, 2.0)
        results[name] = train_model(model, length, epochs)
        print(f"{name:10s} best val acc: {results[name]:.3f}")
    return results


results_30 = compare_models(length=30)

rnn        best val acc: 1.000
lstm       best val acc: 0.512
lstm_bias  best val acc: 1.000
cnn        best val acc: 1.000
attention  best val acc: 1.000


no padding

In [14]:
torch.manual_seed(42)
cnn_valid = BaselineNet("cnn")
cnn_valid.core = nn.Conv1d(32, 32, kernel_size=3, padding=0)   # no padding at the edges
print(train_model(cnn_valid, length=30))

0.6275


## A6. Comparison: RNN vs LSTM vs CNN vs attention

**Task:** random digits 0-9, label = 1 if the FIRST digit is > 4.
The model must remember one early digit until the end of the sequence.

| model | length 10 (val acc) | length 30 (val acc) |
|---|---|---|
| RNN | 1.00 | 1.00 |
| LSTM | 1.00 | 0.50 (stuck) |
| CNN (Conv1D + global max pooling) | 0.80 | 0.63 |
| Attention + pos | 1.00 | 1.00 |

(Keras runs; the length-30 attention result is filled in after running A6.)

**Why each one behaves this way**
- **RNN:** with enough data it can carry the first digit through all steps,
  but the gradient shrinks at every step (vanishing gradient), so it is unstable on long sequences.
- **LSTM:** the forget gate starts around 0.5, so after 29 steps the signal is
  about 0.5**29 ≈ 2e-9. Initializing the forget-gate bias to a positive value
  (e.g. 2.0) fixed it (val acc 1.000).
- **CNN:** global max pooling keeps only the strongest value per filter and drops
  positions, so it cannot say "the first digit".
- **Attention:** the last token looks directly at position 0 (weight ≈ 1.00),
  so distance does not matter. It needs a positional embedding, otherwise
  val acc drops to ~0.62.

**Trade-off:** attention builds an n × n weight table, so memory and compute
grow quadratically with sequence length.

In [12]:
model30, _ = train_attention(use_pos=True, length=30)

Epoch 1/10 | train acc 0.988 | val loss 0.000 acc 1.000
Epoch 2/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 3/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 4/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 5/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 6/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 7/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 8/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 9/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 10/10 | train acc 1.000 | val loss 0.000 acc 1.000
